# FEWS Agent Tools Validation

This notebook validates the tools available in the FEWS Configuration Agent without using the LLM. It directly tests each tool's functionality to ensure they work as expected.

In [1]:
import sys
from pathlib import Path
from IPython.display import display, Markdown

# Add parent directory to path
sys.path.append(Path().absolute().parent.as_posix())

# Import required modules
from talkwithfews.tools import (
    execute_query,
    search_workflows,
    get_workflow_details,
    search_timeseries,
    describe_timeseries,
    search_transformations,
    get_transformation_details,
    analyze_dependencies,
    get_database_schema,
    analyze_dead_points
)

from talkwithfews.models import (
    Deps,
    SQLQueryInput,
    WorkflowSearchInput,
    TimeSeriesSearchInput,
    TransformationSearchInput,
    DependencyAnalysisInput,
    DescribeTimeSeriesInput,
    WorkflowDetailsInput,
    TransformationDetailsInput
)

from talkwithfews.utils import connect_to_database
from talkwithfews.logging_utils import setup_logging
from pydantic_ai import RunContext

# Set logging level to ERROR at the start of the notebook
setup_logging(log_level="ERROR")

In [2]:
# Set up database connection
db_path = '../db/fews_config.db3'
db_connection = connect_to_database(db_path)

# Create dependencies object
deps = Deps(
    db_path=db_path, 
    db_connection=db_connection
)

# Create run context with mock values for required parameters
ctx = RunContext(
    deps=deps,
    model="mock-model",  # Mock model name since we're not using LLM
    usage={              # Mock usage statistics
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0
    },
    prompt="mock-prompt" # Mock prompt since we're not using LLM
)

## 1. Database Schema Validation

In [ ]:
# Test get_database_schema
schema = await get_database_schema(ctx)

# Extract the raw schema
from talkwithfews.schema_utils import extract_db_schema_as_dict, generate_llm_context
raw_schema = extract_db_schema_as_dict(db_path)

# Generate and display the context
context = generate_llm_context(raw_schema)
display(Markdown(context))

## 2. Query Execution Validation

In [ ]:
# Test execute_query with a simple query
simple_query = "SELECT COUNT(*) as count FROM trafos"
result = await execute_query(ctx, SQLQueryInput(query=simple_query))
print("Number of transformations:")
print(result.results[0]['count'])

## 3. Workflow Tools Validation

In [ ]:
# Test search_workflows
search_result = await search_workflows(ctx, WorkflowSearchInput(name_pattern="Import"))
display(Markdown(search_result.markdown_content))

In [ ]:
# Test get_workflow_details
workflow_result = await get_workflow_details(ctx, WorkflowDetailsInput(workflow_id=1))
display(Markdown(workflow_result.markdown_content))

## 4. Time Series Tools Validation

In [ ]:
# Test search_timeseries
search_result = await search_timeseries(ctx, TimeSeriesSearchInput(parameter_id="water"))
display(Markdown(search_result.markdown_content))

In [ ]:
# Test describe_timeseries
desc_result = await describe_timeseries(ctx, DescribeTimeSeriesInput(timeseries_id=123))
display(Markdown(desc_result.markdown_content))

## 5. Transformation Tools Validation

In [ ]:
# Test search_transformations
search_result = await search_transformations(ctx, TransformationSearchInput(description_pattern="water"))
display(Markdown(search_result.markdown_content))

In [ ]:
# Test get_transformation_details
trafo_result = await get_transformation_details(ctx, TransformationDetailsInput(transformation_id=1))
display(Markdown(trafo_result.markdown_content))

## 6. Dependency Analysis Tools Validation

In [ ]:
# Test analyze_dependencies
dep_result = await analyze_dependencies(ctx, DependencyAnalysisInput(entity_type="timeseries", entity_id=123))
display(Markdown(dep_result.markdown_content))

## 7. Dead Points Analysis

In [ ]:
# Test analyze_dead_points
dead_points = await analyze_dead_points(ctx)
Markdown(dead_points.markdown_content)

In [ ]:
dead_points.markdown_content

## Cleanup

In [ ]:
# Close database connection
db_connection.close()